In [2]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

import pandas as pd

from src.utils import Config

In [3]:
from ucimlrepo import fetch_ucirepo 

diabetes_130_us_hospitals_for_years_1999_2008 = fetch_ucirepo(id=296)

X = diabetes_130_us_hospitals_for_years_1999_2008.data.features 
y = diabetes_130_us_hospitals_for_years_1999_2008.data.targets 


/home/lsiau/ai-boilerplate/.venv/lib/python3.12/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


In [22]:

dataset = pd.concat((X, y), axis=1)
dataset.iloc[:,:20].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 20 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   race                      99493 non-null   object
 1   gender                    101766 non-null  object
 2   age                       101766 non-null  object
 3   weight                    3197 non-null    object
 4   admission_type_id         101766 non-null  int64 
 5   discharge_disposition_id  101766 non-null  int64 
 6   admission_source_id       101766 non-null  int64 
 7   time_in_hospital          101766 non-null  int64 
 8   payer_code                61510 non-null   object
 9   medical_specialty         51817 non-null   object
 10  num_lab_procedures        101766 non-null  int64 
 11  num_procedures            101766 non-null  int64 
 12  num_medications           101766 non-null  int64 
 13  number_outpatient         101766 non-null  int64 
 14  numb

In [23]:
cols_to_drop = dataset.columns[dataset.isna().sum() / dataset.shape[0] > 0.25]
print('Cols to drop: ', cols_to_drop.to_list())
dataset_reduced = dataset.drop(cols_to_drop, axis=1)
dataset_reduced.iloc[:,0:20].info()

Cols to drop:  ['weight', 'payer_code', 'medical_specialty', 'max_glu_serum', 'A1Cresult']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 20 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   race                      99493 non-null   object
 1   gender                    101766 non-null  object
 2   age                       101766 non-null  object
 3   admission_type_id         101766 non-null  int64 
 4   discharge_disposition_id  101766 non-null  int64 
 5   admission_source_id       101766 non-null  int64 
 6   time_in_hospital          101766 non-null  int64 
 7   num_lab_procedures        101766 non-null  int64 
 8   num_procedures            101766 non-null  int64 
 9   num_medications           101766 non-null  int64 
 10  number_outpatient         101766 non-null  int64 
 11  number_emergency          101766 non-null  int64 
 12  number_inpatient       

In [24]:
convert_to_obj = ['admission_type_id', "discharge_disposition_id", 'admission_source_id']
dataset_reduced[convert_to_obj] = dataset_reduced[convert_to_obj].astype(object)

In [26]:
dataset_reduced['race'] = dataset_reduced['race'].fillna('Unknown')
dataset_reduced['race'].unique()

array(['Caucasian', 'AfricanAmerican', 'Unknown', 'Other', 'Asian',
       'Hispanic'], dtype=object)

In [29]:
clean_dataset = dataset_reduced.copy()
clean_dataset["readmitted"] = clean_dataset["readmitted"].where(clean_dataset["readmitted"].str.match("<30"), 0).mask(clean_dataset["readmitted"].str.match("<30"), 1)
                                              
clean_dataset['readmitted'].value_counts()


readmitted
0    90409
1    11357
Name: count, dtype: int64

In [30]:
clean_dataset.to_csv(f"../{Config.DATA_DIR}/cleaned_dataset.csv")